
Notebook 2: Dual-Branch Stuttering Classifier — FIXED
======================================================
Reads from Notebook 1 output:
  - X_frame.npy   -> (N, 300, 45)  -> Branch 1: CNN-LSTM
  - X_clip.npy    -> (N, 32)       -> Branch 2: MLP
  - y_labels.npy  -> (N,)          -> Labels
  - metadata.jsonl -> Clip info

Architecture:
  Branch 1 (Temporal):  Conv1D -> BiLSTM -> Attention -> 128-dim embedding
  Branch 2 (Prosody):   MLP(32->64->32->16) -> 16-dim embedding  
  Ensemble:             Concatenate [128; 16] -> FC(144->64->1) + Sigmoid

In [ ]:

# !pip install -q scikit-learn

# %% [code]
import os, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

warnings_ignored = True  # Set to False to see warnings
if warnings_ignored:
    import warnings
    warnings.filterwarnings('ignore')

# ─── SEED ─────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ─── PATHS ──────────────────────────────────────────────────────────────
FEAT_DIR = 'output/features_v3'
OUT_DIR = 'output/working/models'
os.makedirs(OUT_DIR, exist_ok=True)

BATCH_SIZE = 64
EPOCHS = 60
LR = 5e-4 
WD = 1e-3 
WARMUP_EPOCHS = 10
LR_PATIENCE = 10 
LR_FACTOR = 0.5 
EARLY_STOP = 20       # enough room after LR drops

# ─── LOAD FEATURES ────────────────────────────────────────────────────────
print("Loading features from Notebook 1...")

X_frame_path = os.path.join(FEAT_DIR, 'X_frame.npy')
X_clip_path = os.path.join(FEAT_DIR, 'X_clip.npy')
y_path = os.path.join(FEAT_DIR, 'y_labels.npy')
meta_path = os.path.join(FEAT_DIR, 'metadata.jsonl')

# Verify files exist
for p in [X_frame_path, X_clip_path, y_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing: {p}. Run Notebook 1 first.")

X_frame = np.load(X_frame_path)
X_clip = np.load(X_clip_path)
y = np.load(y_path)

print(f"X_frame: {X_frame.shape}  dtype={X_frame.dtype}")
print(f"X_clip:  {X_clip.shape}  dtype={X_clip.dtype}")
print(f"y:       {y.shape}  dtype={y.dtype}")

# Infer dimensions
N_CLIPS, N_TIME_FRAMES, N_FRAME_FEAT = X_frame.shape
N_CLIP_FEAT = X_clip.shape[1]

assert X_clip.shape[0] == N_CLIPS, "X_frame and X_clip have different N"
assert len(y) == N_CLIPS, "Labels count mismatch"

# ─── LOAD METADATA (optional, for analysis) ─────────────────────────────
meta_list = []
if os.path.exists(meta_path):
    with open(meta_path, 'r') as f:
        for line in f:
            meta_list.append(json.loads(line.strip()))
    print(f"Metadata: {len(meta_list)} clips loaded")

# ─── TRAIN/VAL/TEST SPLIT ───────────────────────────────────────────────
# Stratified split preserving class ratio
# First: separate features and labels
indices = np.arange(N_CLIPS)

# 70/15/15 split
idx_train, idx_temp = train_test_split(
    indices, test_size=0.40, stratify=y
)

idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, stratify=y[idx_temp]
)


# Extract subsets
X_frame_train, X_frame_val, X_frame_test = X_frame[idx_train], X_frame[idx_val], X_frame[idx_test]
X_clip_train, X_clip_val, X_clip_test = X_clip[idx_train], X_clip[idx_val], X_clip[idx_test]
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]

print(f"\n{'='*50}")
print(f"Data splits:")
print(f"  Train: {len(y_train)}  (stutter={(y_train==1).sum()}, clean={(y_train==0).sum()})")
print(f"  Val:   {len(y_val)}    (stutter={(y_val==1).sum()}, clean={(y_val==0).sum()})")
print(f"  Test:  {len(y_test)}   (stutter={(y_test==1).sum()}, clean={(y_test==0).sum()})")

# ─── DEVICE ─────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDevice: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# %%



In [ ]:



class StutterDataset(Dataset):
    def __init__(self, X_frame, X_clip, y):
        # Replace NaN/Inf with 0 before converting to tensor
        X_frame = np.nan_to_num(X_frame, nan=0.0, posinf=0.0, neginf=0.0)
        X_clip  = np.nan_to_num(X_clip,  nan=0.0, posinf=0.0, neginf=0.0)

        self.X_frame = torch.tensor(X_frame, dtype=torch.float32)
        self.X_clip  = torch.tensor(X_clip,  dtype=torch.float32)
        self.y       = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_frame[idx], self.X_clip[idx], self.y[idx]



# 70/15/15 split
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, stratify=y[idx_temp]
)

idx_train, idx_temp = train_test_split(
    indices, test_size=0.40, stratify=y
)


# Extract subsets
X_frame_train, X_frame_val, X_frame_test = X_frame[idx_train], X_frame[idx_val], X_frame[idx_test]
X_clip_train, X_clip_val, X_clip_test = X_clip[idx_train], X_clip[idx_val], X_clip[idx_test]
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]

print(f"\n{'='*50}")
print(f"Data splits:")
print(f"  Train: {len(y_train)}  (stutter={(y_train==1).sum()}, clean={(y_train==0).sum()})")
print(f"  Val:   {len(y_val)}    (stutter={(y_val==1).sum()}, clean={(y_val==0).sum()})")
print(f"  Test:  {len(y_test)}   (stutter={(y_test==1).sum()}, clean={(y_test==0).sum()})")

# ─── DEVICE ─────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDevice: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")



train_ds = StutterDataset(X_frame_train, X_clip_train, y_train)
val_ds = StutterDataset(X_frame_val, X_clip_val, y_val)
test_ds = StutterDataset(X_frame_test, X_clip_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, 
                          num_workers=2, pin_memory=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, 
                        num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, 
                         num_workers=2, pin_memory=True)

print(f"Dataloaders ready:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches:   {len(val_loader)}")
print(f"  Test batches:  {len(test_loader)}")



In [ ]:



class TemporalBranch(nn.Module):
    """
    Branch 1: CNN-LSTM on frame-level features.
    Input:  (batch, time=300, features=45)
    Output: (batch, 128) clip embedding
    """
    def __init__(self, n_time=300, n_feat=45, hidden=64, out_dim=128):
        super().__init__()

        # Conv1D: processes (batch, features, time)
        self.conv = nn.Sequential(
            nn.Conv1d(n_feat, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),

            nn.Conv1d(128, hidden, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),
        )

        # After 2 MaxPool1d(2): time -> 300/4 = 75
        conv_time = n_time // 4

        # BiLSTM
        self.lstm = nn.LSTM(
            input_size=hidden,
            hidden_size=hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        # Self-attention
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden * 2,
            num_heads=4,
            batch_first=True
        )

        # Projection to embedding
        self.fc = nn.Sequential(
            nn.Linear(hidden * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, out_dim)
        )

    def forward(self, x):
        # x: (batch, time, feat) -> (batch, feat, time) for Conv1d
        x = x.permute(0, 2, 1)
        x = self.conv(x)           # (batch, hidden, time/4)
        x = x.permute(0, 2, 1)     # (batch, time/4, hidden)

        # LSTM
        x, _ = self.lstm(x)        # (batch, time/4, hidden*2)

        # Attention pooling
        attn_out, attn_weights = self.attention(x, x, x)
        x = attn_out.mean(dim=1)   # (batch, hidden*2)

        # Projection
        x = self.fc(x)             # (batch, out_dim)
        return x


class ProsodyBranch(nn.Module):
    """
    Branch 2: MLP on clip-level derived features.
    Input:  (batch, 32)
    Output: (batch, 16) prosody embedding
    """
    def __init__(self, in_dim=32, hidden1=64, hidden2=32, out_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(hidden2, out_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class StutterClassifier(nn.Module):
    """
    Dual-branch ensemble classifier.
    """
    def __init__(self, temporal_dim=128, prosody_dim=16):
        super().__init__()
        self.temporal = TemporalBranch(
            n_time=N_TIME_FRAMES, 
            n_feat=N_FRAME_FEAT, 
            out_dim=temporal_dim
        )
        self.prosody = ProsodyBranch(
            in_dim=N_CLIP_FEAT, 
            out_dim=prosody_dim
        )

        # Fusion classifier
        fusion_dim = temporal_dim + prosody_dim
        self.fusion = nn.Sequential(
            nn.Linear(fusion_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 1)
        )

    def forward(self, x_frame, x_clip):
        emb_t = self.temporal(x_frame)    # (batch, 128)
        emb_p = self.prosody(x_clip)       # (batch, 16)
        emb = torch.cat([emb_t, emb_p], dim=1)  # (batch, 144)
        return self.fusion(emb)             # (batch, 1) — logits


# %% [code]
print("NaN check:")
print(f"  X_frame NaNs: {np.isnan(X_frame).sum()} ({np.isnan(X_frame).mean()*100:.2f}%)")
print(f"  X_clip  NaNs: {np.isnan(X_clip).sum()} ({np.isnan(X_clip).mean()*100:.2f}%)")
# Find which clip-level features have NaNs
nan_cols = np.where(np.isnan(X_clip).any(axis=0))[0]
print(f"  X_clip NaN columns: {nan_cols}")




In [ ]:



model = StutterClassifier(temporal_dim=128, prosody_dim=16).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {total_params:,}")

# Class imbalance handling
n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()

pos_weight = torch.tensor([n_neg / max(n_pos, 1) * 2], dtype=torch.float32).to(DEVICE)
print(f"Class weights: pos={pos_weight.item():.2f} (n_pos={n_pos}, n_neg={n_neg})")

# Loss: BCEWithLogitsLoss (more stable than BCE + Sigmoid)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Optimizer: AdamW with weight decay
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=LR_FACTOR, patience=LR_PATIENCE)



In [ ]:



def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for x_frame, x_clip, labels in loader:
        x_frame = x_frame.to(DEVICE)
        x_clip = x_clip.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(x_frame, x_clip)
        loss = criterion(outputs, labels)
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item() * len(labels)
        all_preds.extend(torch.sigmoid(outputs).detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, np.array(all_preds), np.array(all_labels)


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x_frame, x_clip, labels in loader:
            x_frame = x_frame.to(DEVICE)
            x_clip = x_clip.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(x_frame, x_clip)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * len(labels)

            all_preds.extend(torch.sigmoid(outputs).detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, np.array(all_preds), np.array(all_labels)


def compute_metrics(y_true, y_prob, threshold=0.5):
    """Compute classification metrics."""
    y_pred = (y_prob > threshold).astype(int)

    acc = (y_pred == y_true).mean()
    auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.5
    f1 = f1_score(y_true, y_pred, zero_division=0)

    return {
        'accuracy': acc,
        'auc': auc,
        'f1': f1,
        'precision': (y_pred[y_true==1] == 1).mean() if (y_true==1).sum() > 0 else 0,
        'recall': (y_pred[y_true==1] == 1).sum() / max((y_true==1).sum(), 1),
    }


# ─── TRAINING ───────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"STARTING TRAINING")
print(f"{'='*60}")

best_auc = 0
best_state = None
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_acc': [], 'val_f1': []}

for epoch in range(1, EPOCHS + 1):
    # Train
    train_loss, train_preds, train_labels = train_epoch(model, train_loader, criterion, optimizer)

    # Validate
    val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion)
    
    # Metrics
    val_metrics = compute_metrics(val_labels, val_preds)
    train_metrics = compute_metrics(train_labels, train_preds)

    # Scheduler step
    scheduler.step(val_metrics['auc'])

    # History
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_metrics['auc'])
    history['val_acc'].append(val_metrics['accuracy'])
    history['val_f1'].append(val_metrics['f1'])

    # Print
    if epoch % 10 == 0 : 
        print(f"Epoch {epoch:3d}/{EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val AUC: {val_metrics['auc']:.4f} | "
              f"Val F1: {val_metrics['f1']:.4f} | "
              f"Val Acc: {val_metrics['accuracy']:.4f}")

    # Early stopping
    if val_metrics['auc'] > best_auc:
        best_auc = val_metrics['auc']
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        # Save best
        torch.save(best_state, os.path.join(OUT_DIR, 'best_model.pt'))
    else:
        patience_counter += 1

    if patience_counter >= EARLY_STOP:
        print(f"\nEarly stopping at epoch {epoch}. Best Val AUC: {best_auc:.4f}")
        break

# Load best model
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"\nLoaded best model (Val AUC: {best_auc:.4f})")



In [ ]:



print(f"\n{'='*60}")
print(f"TEST SET EVALUATION")
print(f"{'='*60}")

test_loss, test_preds, test_labels = evaluate(model, test_loader, criterion)
test_metrics = compute_metrics(test_labels, test_preds)

print(f"Test Loss:    {test_loss:.4f}")
print(f"Test AUC:     {test_metrics['auc']:.4f}")
print(f"Test F1:      {test_metrics['f1']:.4f}")
print(f"Test Acc:     {test_metrics['accuracy']:.4f}")
print(f"Test Recall:  {test_metrics['recall']:.4f}")
print(f"Test Precision: {test_metrics['precision']:.4f}")

# Classification report
y_pred = (test_preds > 0.5).astype(int)
print(f"\nClassification Report:")
print(classification_report(test_labels, y_pred, 
                            target_names=['No Stutter', 'Stutter'],
                            digits=4))

# %%


In [ ]:



fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Training curves
ax = axes[0, 0]
ax.plot(history['train_loss'], label='Train Loss', color='blue')
ax.plot(history['val_loss'], label='Val Loss', color='orange')
ax.set_title('Loss Curves')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. AUC curve
ax = axes[0, 1]
ax.plot(history['val_auc'], label='Val AUC', color='green', lw=2)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Validation AUC')
ax.set_xlabel('Epoch')
ax.set_ylabel('AUC')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Accuracy & F1
ax = axes[0, 2]
ax.plot(history['val_acc'], label='Accuracy', color='purple')
ax.plot(history['val_f1'], label='F1 Score', color='red')
ax.set_title('Validation Metrics')
ax.set_xlabel('Epoch')
ax.set_ylabel('Score')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Confusion Matrix
ax = axes[1, 0]
cm = confusion_matrix(test_labels, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100  # row-normalize

sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', ax=ax,
            xticklabels=['No Stutter', 'Stutter'],
            yticklabels=['No Stutter', 'Stutter'],
            vmin=0, vmax=100)
ax.set_title('Confusion Matrix (%)')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')

# 5. ROC Curve
ax = axes[1, 1]
fpr, tpr, _ = roc_curve(test_labels, test_preds)
ax.plot(fpr, tpr, label=f'Test AUC = {test_metrics["auc"]:.3f}', lw=2, color='darkorange')
ax.plot([0, 1], [0, 1], '--', color='gray', alpha=0.5)
ax.set_title('ROC Curve')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

# 6. Prediction distribution
ax = axes[1, 2]
ax.hist(test_preds[test_labels==0], bins=30, alpha=0.6, label='No Stutter', color='blue')
ax.hist(test_preds[test_labels==1], bins=30, alpha=0.6, label='Stutter', color='red')
ax.axvline(x=0.5, color='black', linestyle='--', label='Threshold')
ax.set_title('Prediction Distribution')
ax.set_xlabel('Predicted Probability')
ax.set_ylabel('Count')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Stuttering Classifier — Training & Evaluation Results', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'training_results.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPlot saved: training_results.png")



In [ ]:



# Save full model (architecture + weights)
torch.save(model, os.path.join(OUT_DIR, 'stutter_classifier_full.pt'))

# Save just weights
torch.save(model.state_dict(), os.path.join(OUT_DIR, 'stutter_classifier_weights.pt'))

# Save training history
with open(os.path.join(OUT_DIR, 'history.json'), 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.items()}, f, indent=2)

# Save model summary
model_summary = {
    "architecture": "Dual-Branch CNN-LSTM + MLP Ensemble",
    "input_shapes": {
        "X_frame": list(X_frame.shape),
        "X_clip": list(X_clip.shape),
    },
    "branches": {
        "temporal": {
            "input": f"(batch, {N_TIME_FRAMES}, {N_FRAME_FEAT})",
            "layers": ["Conv1D(45->64->128->128)", "BiLSTM(128, 2-layers)", "MultiheadAttention", "FC(256->128)"],
            "output_dim": 128,
        },
        "prosody": {
            "input": f"(batch, {N_CLIP_FEAT})",
            "layers": ["FC(32->64->32->16)"],
            "output_dim": 16,
        },
        "fusion": {
            "input": "Concatenate [128; 16] = 144",
            "layers": ["FC(144->64->1)"],
            "output": "logits (apply sigmoid for probability)",
        },
    },
    "training": {
        "epochs_trained": len(history['train_loss']),
        "best_val_auc": float(best_auc),
        "test_auc": float(test_metrics['auc']),
        "test_f1": float(test_metrics['f1']),
        "test_accuracy": float(test_metrics['accuracy']),
    },
    "parameters": total_params,
    "hyperparameters": {
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "weight_decay": WD,
        "pos_weight": float(pos_weight.item()),
    },
}

with open(os.path.join(OUT_DIR, 'model_summary.json'), 'w') as f:
    json.dump(model_summary, f, indent=2)

print(f"\n{'='*60}")
print(f"SAVED TO: {OUT_DIR}")
print(f"{'='*60}")
for fn in ['best_model.pt', 'stutter_classifier_full.pt', 
           'stutter_classifier_weights.pt', 'history.json', 
           'model_summary.json', 'training_results.png']:
    fp = os.path.join(OUT_DIR, fn)
    if os.path.exists(fp):
        size_mb = os.path.getsize(fp) / 1e6
        print(f"  {fn:30s}  {size_mb:6.2f} MB")

print(f"\n✅ Training complete. Best Val AUC: {best_auc:.4f} | Test AUC: {test_metrics['auc']:.4f}")

# %%
import torch, os, datetime

checkpoint = {
    'model_state_dict': model.state_dict(),
    'timestamp': datetime.datetime.now().isoformat(),
}


save_path = os.path.join(OUT_DIR, f"stutter_model_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.pth")
torch.save(checkpoint, save_path)
print(f"Saved: {save_path} ({os.path.getsize(save_path)/1e6:.2f} MB)")


